In [13]:
config = {
    "dataset": "wine",
    "models": ["modernnca", "catboost", "xgboost"],  # Можно добавить "mlp"
    "seeds": [0],  # Для каждого seed создаётся новое разбиение 64/16/20
    "tune_seed": 0,
    "n_trials": 10,  # На каждую модель и каждое разбиение
    "max_epochs": 200,
    "patience": 20,  # Эпох без улучшения для нейросетей
    "max_boost_rounds": 2000,
    "early_stopping_rounds": 50,  # Итераций без улучшения для бустов
    "batch_size": 1024,
    "modernnca": {
        "n_frequencies": 77,
    },
}

In [14]:
from Stream1_DataExtraction import *

dataset = load_dataset(config["dataset"], seed=config["seeds"][0])

print(dataset.categorical_columns)

[]


In [15]:
from Stream2_Preprocessing import *

prep = TabularPreprocessor(dataset.numerical_columns, dataset.categorical_columns)
prep.fit(dataset.X_train)

In [16]:
train = prep.transform(dataset.X_train, dataset.y_train)

In [17]:
test = prep.transform(dataset.X_test, dataset.y_test)

In [18]:
val = prep.transform(dataset.X_val, dataset.y_val)

In [19]:
runs = [(config["seeds"][0], dataset, prep, train, val, test)]

for seed in config["seeds"][1:]:
    dataset_run = load_dataset(config["dataset"], seed=seed)
    prep_run = TabularPreprocessor(dataset_run.numerical_columns, dataset_run.categorical_columns)
    prep_run.fit(dataset_run.X_train)
    train_run = prep_run.transform(dataset_run.X_train, dataset_run.y_train)
    val_run = prep_run.transform(dataset_run.X_val, dataset_run.y_val)
    test_run = prep_run.transform(dataset_run.X_test, dataset_run.y_test)
    runs.append((seed, dataset_run, prep_run, train_run, val_run, test_run))

In [20]:
from Stream3_ModernNCA import *


def build_modern_nca(params, seed, train):
    torch.manual_seed(seed)
    n_num = train.x_num.shape[1] if train.x_num is not None else 0
    n_cat = train.x_cat.shape[1] if train.x_cat is not None else 0
    embedder = PLREmbeddings(n_num, n_frequencies=params["n_frequencies"], d_embedding=params["plr_dim"]) if n_num else None

    return ModernNCAEncoder(
        plr_module=embedder,
        num_continuous=n_num,
        plr_dim=params["plr_dim"],
        num_categorical=n_cat,
        hidden_dim=params["hidden_dim"],
        embed_dim=params["embed_dim"],
        num_layers=params["num_layers"],
        dropout_rate=params["dropout"],
        post_mlp_layers=[],
    )

In [21]:
from Stream4_Train import *
import optuna

nca_runs = []

if "modernnca" in config["models"]:
    for seed, dataset_run, prep_run, train_run, val_run, test_run in runs:
        train_data = (train_run.x_num, train_run.x_cat, train_run.y)
        val_data = (val_run.x_num, val_run.x_cat, val_run.y)
        nca_params = {
            **config["modernnca"],
            "batch_size": config["batch_size"],
            "patience": config["patience"],
            "verbose": False,
        }

        def nca_objective(trial):
            params = {
                **nca_params,
                "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
                "dropout": trial.suggest_float("dropout", 0.0, 0.3),
                "hidden_dim": trial.suggest_categorical("hidden_dim", [32, 64, 128]),
                "embed_dim": trial.suggest_categorical("embed_dim", [32, 64, 128]),
                "plr_dim": trial.suggest_categorical("plr_dim", [16, 34, 64]),
                "num_layers": trial.suggest_int("num_layers", 1, 2),
                "sample_rate": trial.suggest_float("sample_rate", 0.1, 0.5),
            }
            trial.set_user_attr("config", params)
            trainer_trial = Trainer(build_modern_nca(params, config["tune_seed"], train_run), params)
            trainer_trial.fit(train_data, config["max_epochs"], params["sample_rate"], dataset_run.task_type, dataset_run.n_classes, val_data)
            return trainer_trial.best_score

        study = optuna.create_study(
            direction="minimize" if dataset_run.task_type == "regression" else "maximize",
            sampler=optuna.samplers.TPESampler(seed=config["tune_seed"]),
        )
        study.optimize(nca_objective, n_trials=config["n_trials"])
        nca_params = study.best_trial.user_attrs["config"].copy()

        model = build_modern_nca(nca_params, seed, train_run)
        trainer = Trainer(model, nca_params)
        trainer.fit(train_data, config["max_epochs"], nca_params["sample_rate"], dataset_run.task_type, dataset_run.n_classes, val_data)
        nca_runs.append((seed, dataset_run, train_run, test_run, trainer, nca_params))

[I 2026-10-05 17:03:54,038] A new study created in memory with name: no-name-451cd983-3800-4e3a-95e5-9fc158c2cbb5
[I 2026-10-05 17:03:58,835] Trial 0 finished with value: 0.7965686274509803 and parameters: {'lr': 0.0012520653814999472, 'weight_decay': 0.0001398196140899405, 'dropout': 0.18082901282149316, 'hidden_dim': 128, 'embed_dim': 128, 'plr_dim': 34, 'num_layers': 2, 'sample_rate': 0.47023865531706444}. Best is trial 0 with value: 0.7965686274509803.
[I 2026-10-05 17:04:01,093] Trial 1 finished with value: 0.7622549019607843 and parameters: {'lr': 0.00013869861245357332, 'weight_decay': 1.8255254802399014e-06, 'dropout': 0.006065519232097715, 'hidden_dim': 128, 'embed_dim': 32, 'plr_dim': 16, 'num_layers': 1, 'sample_rate': 0.47786756681983356}. Best is trial 0 with value: 0.7965686274509803.
[I 2026-10-05 17:04:02,246] Trial 2 finished with value: 0.7696078431372549 and parameters: {'lr': 0.001105851072569646, 'weight_decay': 1.7538232373118042e-05, 'dropout': 0.0793666836313880

In [22]:
for seed, dataset_run, train_run, test_run, trainer, params in nca_runs:
    print(seed, "best epoch", trainer.best_epoch, "epochs trained", trainer.epochs_trained)

0 best epoch 34 epochs trained 54


In [23]:
from steram5 import compute_metric

nca_results = []

for seed, dataset_run, train_run, test_run, trainer, params in nca_runs:
    predictions = trainer.predict(
        (train_run.x_num, train_run.x_cat, train_run.y),
        (test_run.x_num, test_run.x_cat),
        dataset_run.task_type,
        dataset_run.n_classes,
    )

    if dataset_run.task_type in ("binary", "multiclass"):
        predictions = predictions.argmax(dim=1)

    metric_name, metric_value = compute_metric(
        dataset_run.task_type,
        test_run.y.cpu().numpy(),
        predictions.cpu().numpy(),
    )
    nca_results.append({
        "dataset": dataset_run.dataset_name,
        "model": "modernnca",
        "seed": seed,
        "task_type": dataset_run.task_type,
        "metric": metric_name,
        "metric_value": metric_value,
        "config": params.copy(),
    })
    print(seed, metric_name, metric_value)

0 accuracy 0.76171875


In [24]:
from steram5 import *

baseline_names = [name for name in config["models"] if name != "modernnca"]
baseline_configs = {
    "catboost": {"iterations": config["max_boost_rounds"], "early_stopping_rounds": config["early_stopping_rounds"]},
    "xgboost": {"n_estimators": config["max_boost_rounds"], "early_stopping_rounds": config["early_stopping_rounds"]},
    "mlp": {"max_epoch": config["max_epochs"], "patience": config["patience"], "batch_size": config["batch_size"]},
}
frames = [pd.DataFrame(nca_results, columns=["dataset", "model", "seed", "task_type", "metric", "metric_value", "config"])]

if baseline_names:
    for seed, dataset_run, prep_run, train_run, val_run, test_run in runs:
        frames.append(run_experiments(
            model_names=baseline_names,
            datasets=[dataset_run],
            seeds=[seed],
            preprocessor=prep_run,
            n_trials=config["n_trials"],
            tune_seed=config["tune_seed"],
            model_configs=baseline_configs,
        ))

results = pd.concat(frames, ignore_index=True)
print(results[["dataset", "model", "seed", "metric", "metric_value"]])

[I 2026-10-05 17:04:18,910] A new study created in memory with name: no-name-d043c524-5e9b-481e-ba19-85f67d011623
[I 2026-10-05 17:04:19,360] Trial 0 finished with value: 0.7598039215686274 and parameters: {'bagging_temperature': 0.5488135039273248, 'depth': 8, 'l2_leaf_reg': 4.006483661964395, 'leaf_estimation_iterations': 6, 'learning_rate': 0.0013130280280658588}. Best is trial 0 with value: 0.7598039215686274.
[I 2026-10-05 17:04:19,533] Trial 1 finished with value: 0.7573529411764706 and parameters: {'bagging_temperature': 0.6458941130666561, 'depth': 6, 'l2_leaf_reg': 7.79422611516095, 'leaf_estimation_iterations': 10, 'learning_rate': 0.0008264328927007723}. Best is trial 0 with value: 0.7598039215686274.
[I 2026-10-05 17:04:19,731] Trial 2 finished with value: 0.7622549019607843 and parameters: {'bagging_temperature': 0.7917250380826646, 'depth': 7, 'l2_leaf_reg': 3.6986612821028797, 'leaf_estimation_iterations': 10, 'learning_rate': 2.2655846366566637e-05}. Best is trial 2 wit

  dataset      model  seed    metric  metric_value
0    wine  modernnca     0  accuracy      0.761719
1    wine   catboost     0  accuracy      0.763672
2    wine    xgboost     0  accuracy      0.791016
